In [ ]:
import pandas as pd
import numpy as np
import pickle
import numpy
from sklearn.metrics.pairwise import cosine_similarity
from scipy import sparse

In [17]:
selected = ['Toy Story', "Jumanji", "Toy Story 2", "Batman", "Batman Forever"]

In [10]:
df = pd.read_csv('display_movie_data.csv')

In [18]:
df[df['title'].isin(selected)]['MovieID'].values

array([   1,    2,  153,  592, 3114])

In [ ]:
with open('cb_movie_to_id.pkl', 'rb') as file:
    cb_movie_to_id = pickle.load(file)
    file.close()
    
with open('cb_id_to_movie.pkl', 'rb') as file:
    cb_id_to_movie = pickle.load(file)
    file.close()
    
tfidf_matrix = sparse.load_npz('models/artifacts/matrices/tfidf_matrix.npz')

In [20]:
cb_movie_to_id[1]

0

In [23]:
tfidf_matrix[[1,2,3]]

array([[0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.],
       [0., 0., 0., ..., 0., 0., 0.]], shape=(3, 45119))

In [ ]:
def cold_start_recommender(selected_movies, num_movies=5):
    """
    Cold-start recommender for users without prior ratings.
    Takes a list of selected movie titles, constructs a user preference
    vector by averaging their TF-IDF representations, and recommends
    similar movies using cosine similarity.

    """
    
    movie_ids  = df[df['title'].isin(selected_movies)]['MovieID'].values
        
    movie_idxs = [cb_movie_to_id[i] for i in movie_ids if i in cb_movie_to_id]
    
    if not movie_idxs:
        return None
    
    # cosine similarity expects a 2d array, our mean_vector is current 1d -> [1,2,3,..]
    
    mean_vector = tfidf_matrix[movie_idxs].mean(axis=0) # converting [1,2] -> [[1,2]] 
    mean_vector = np.asarray(mean_vector).reshape(1,-1)


    # 2d vector
    similiaritiess = cosine_similarity(mean_vector, tfidf_matrix).flatten()

    # sorts the index. like argmin() return id of min
    most_similar_movies = similiaritiess.argsort()
    
    # Remove selected movies
    most_similar_movies = most_similar_movies[~np.isin(most_similar_movies, movie_idxs)]

    recommend_movies = most_similar_movies[::-1]
    
    
    # converting idx to MovieId
    recommend_movies = [ cb_id_to_movie[int(i)] for i in recommend_movies if int(i) in cb_id_to_movie][:num_movies]

    return df.loc[df['MovieID'].isin(recommend_movies),:]
    
    
    
    
    
    


In [28]:
cold_start_recommender(selected,30)

,MovieID,title,release_year,original_language,genre,overview,production_companies,keywords,cast,director,poster_url
2,3,Grumpier Old Men,1995,en,"romance, comedy",A family wedding reignites the ancient feud be...,"Lancaster Gate, Warner Bros. Pictures","fishing, sequel, old man, best friend, wedding...","Walter Matthau, Jack Lemmon, Ann-Margret, Soph...",Howard Deutch,https://image.tmdb.org/t/p/w500/1FSXpj5e8l4KH6...
129,135,Down Periscope,1996,en,comedy,Maverick Navy Lieutenant Commander Tom Dodge w...,20th Century Fox,"mutiny, submarine, u.s. navy, misfit, maverick...","Kelsey Grammer, Lauren Holly, Rob Schneider, H...",David S. Ward,https://image.tmdb.org/t/p/w500/brleKcHsirTw7l...
153,161,Crimson Tide,1995,en,"war, action, drama, thriller","After the Cold War, a breakaway Russian republ...","Hollywood Pictures, Don Simpson/Jerry Bruckhei...","mutiny, submarine, missile, embassy, nuclear m...","Denzel Washington, Gene Hackman, Matt Craven, ...",Tony Scott,https://image.tmdb.org/t/p/w500/21nqRJ6ofEgVvE...
380,392,The Secret Adventures of Tom Thumb,1993,en,"science fiction, animation, horror, children's...",A boy born the size of a small doll is kidnapp...,"Lumen Films, Tara, Bolex Brothers, BBC Bristol...",stop motion,"Nick Upton, Deborah Collard, Frank Passingham,...",Dave Borthwick,https://image.tmdb.org/t/p/w500/98yogK8pBvjqpQ...
541,558,The Pagemaster,1994,en,"animation, adventure, children's, family, fant...","Rich knows a lot about accidents. So much so, ...","20th Century Fox, Turner Pictures, David Kirsc...","rain, animated scene, bike, live action and an...","Macaulay Culkin, Christopher Lloyd, Whoopi Gol...","Joe Johnston, Pixote Hunt",https://image.tmdb.org/t/p/w500/iIWUuCjN8VND7B...
592,610,Heavy Metal,1981,en,"music, science fiction, animation, horror, adv...","The embodiment of ultimate evil, a glowing orb...","Canadian Film Development Corporation, Columbi...","flying car, taxi, heavy metal, based on comic,...","Rodger Bumpass, John Candy, Jackie Burroughs, ...","Gerald Potterton, John Bruno, John Halas, Juli...",https://image.tmdb.org/t/p/w500/atUtWrDlLzT1ye...
609,631,All Dogs Go to Heaven 2,1996,en,"musical, animation, adventure, children's, fam...",Charlie and Itchy return to Earth to find Gabr...,Metro-Goldwyn-Mayer Animation,"angel, san francisco, california, villain, hea...","Charlie Sheen, Dom DeLuise, Adam Wylie, Sheena...","Larry Leker, Paul Sabella",https://image.tmdb.org/t/p/w500/kNmIJILOW9qF2F...
810,870,Gone Fishin',1997,en,comedy,Two fishing fanatics get in trouble when their...,"Hollywood Pictures, Caravan Pictures, Roger Bi...","sea, boat, fishing, rogue","Joe Pesci, Danny Glover, Rosanna Arquette, Lyn...",Christopher Cain,https://image.tmdb.org/t/p/w500/lGRHE7s2iGgJ0J...
1046,1127,The Abyss,1989,en,"thriller, science fiction, adventure, sci-fi, ...",A civilian oil rig crew is recruited to conduc...,"20th Century Fox, Pacific Western","sea, flying saucer, submarine, ocean, diving s...","Ed Harris, Mary Elizabeth Mastrantonio, Michae...",James Cameron,https://image.tmdb.org/t/p/w500/2dCit3XAtv9KWC...
1110,1205,The Transformers: The Movie,1986,en,"thriller, war, science fiction, animation, adv...",The Autobots must stop a colossal planet-consu...,"Marvel Productions, Sunbow Productions","transformation, based on cartoon, based on toy...","Judd Nelson, Peter Cullen, Frank Welker, Leona...",Nelson Shin,https://image.tmdb.org/t/p/w500/xJ5W0mOAWCVaXm...


In [ ]:
for i in range(0,5,5):
    for j in range(i,i+5):
        print(j)

0
1
2
3
4
5
6
7
8
9
10
11
12
13
14


In [ ]:
for i in range(0, len(movies)+1, 5):